# visualization_004

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (140).ipynb`

| Field | Value |
|---|---|
| Section | `visualization` |
| Market | `indian_equities` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 23 |
| Detected functions | save_plot_as_image, plot_cumulative_pnl_and_drawdown, plot_drawdown, plot_pnl_distribution, plot_monthly_profit_heatmap, plot_yearly_returns, plot_quarterly_profit |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import pandas_datareader as pdr
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
file = pd.read_parquet('OCT/17102025_spot_tick.parquet')
file.Ticker.unique()

In [ ]:
import pandas as pd
# Now you can load the entire rows with ICICIBANK.NSE if required
BuyPrice_Stock1 = pd.read_csv('ICICIBANK_OCT2024.csv')
BuyPrice_Stock1

In [ ]:
import pandas as pd
BuyPrice_Stock2 = pd.read_csv('LT_OCT2024.csv')
BuyPrice_Stock2

In [ ]:
import pandas as pd
# Assuming df1 and df2 are your two DataFrames and 'Date_Time' is the common column
combined_df = pd.merge(BuyPrice_Stock1, BuyPrice_Stock2, on='Date_Time', how='inner')  # 'inner' join will keep only the rows where 'Date_Time' is common in both datasets
combined_df.head()

In [ ]:
import pandas as pd
# Assuming combined_df is already defined and 'data' DataFrame is created
data = pd.DataFrame({
'DateTime': combined_df['Date_Time'],
'BuyPrice_Stock1': combined_df['BuyPrice_y'],
'BuyPrice_Stock2': combined_df['BuyPrice_x'],
'SellPrice_Stock1': combined_df['SellPrice_y'],
'SellPrice_Stock2': combined_df['SellPrice_x']
})
# Remove rows where any price column has a 0 value
data = data[(data[['BuyPrice_Stock1', 'BuyPrice_Stock2', 'SellPrice_Stock1', 'SellPrice_Stock2']] != 0).all(axis=1)]
# Display the resulting DataFrame
data

In [ ]:
# Ensure 'DateTime' is in datetime format
data['DateTime'] = pd.to_datetime(data['DateTime'])
# Plotting
plt.figure(figsize=(12, 6))
plt.plot(data['DateTime'], data['BuyPrice_Stock1'], label='Buy Price Stock1', color='green')
plt.plot(data['DateTime'], data['BuyPrice_Stock2'], label='Buy Price Stock2', color='red')
# Formatting the plot
plt.xlabel('DateTime')
plt.ylabel('Price')
plt.title('Buy and Sell Prices of Stock1 and Stock2')
plt.legend(loc='upper left')
plt.xticks(rotation=45)
plt.grid(True)
# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
# Create the figure
fig = plt.figure(figsize=(13, 7))
# Calculate the difference between the 'close' prices of BuyPrice_Stock1.NS and BuyPrice_Stock2.NS
price_difference_Buy = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) / (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])
# Plot the difference
plt.plot(price_difference_Buy, label='Buy Price Difference')
plt.legend()
plt.title("Difference in Prices: Stock2 - Stock1")
plt.xlabel("Date")
plt.ylabel("Price Difference")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
# Calculate the ratio of HdataC to ICICI
ratio = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) / (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])
ratio_mean = ratio.rolling(window=5500).mean()
# Create the figure and plot the ratio
fig = plt.figure(figsize=(13, 7))
plt.plot(ratio, label='BuyPrice_Stock1 / BuyPrice_Stock2')
# Add a horizontal line at the mean of the ratio
plt.plot(ratio_mean, color='red', linestyle='--', label='Mean Ratio')
# Add legend, title, and labels
plt.legend()
plt.title("Ratio of BuyPrice_Stock1 to BuyPrice_Stock2 close Prices")
plt.xlabel("Date")
plt.ylabel("Ratio")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Calculate the ratio of BuyPrice_Stock1 to BuyPrice_Stock2
ratio = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) / (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])
# Calculate the 43-min moving average
moving_average_43 = ratio.rolling(window=4300).mean()
# Calculate the mean and standard deviation of the ratio
mean_ratio = ratio.mean()
std_ratio = ratio.std()
# Calculate the Z-score using the ratio
z_score = (ratio - moving_average_43) / moving_average_43.std()
# Create the figure and plot the Z-score
fig = plt.figure(figsize=(13, 7))
plt.plot(z_score, label='Z-Score of BuyPrice_Stock1 / BuyPrice_Stock2', color='purple')
# Add horizontal lines for Z-scores of -1 and 1 for reference
plt.axhline(0.2, color='green', linestyle='--', label='Z-Score = 0.2')
plt.axhline(-0.2, color='green', linestyle='--', label='Z-Score = -0.2')
# Add buy and sell markers
buy_signals = z_score[z_score < -0.2]
sell_signals = z_score[z_score > 0.2]
# Mark buy signals
plt.scatter(buy_signals.index, buy_signals, marker='^', color='green', label='Buy Signal', s=100)
# Mark sell signals
plt.scatter(sell_signals.index, sell_signals, marker='v', color='red', label='Sell Signal', s=100)
# Add legend, title, and labels
plt.legend()
plt.title("Z-Score of BuyPrice_Stock1 to BuyPrice_Stock2 Ratio with Buy/Sell Signals")
plt.xlabel("Date")
plt.ylabel("Z-Score")
plt.show()

In [ ]:
data['ratio'] = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) / (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])
data['difference'] = data['BuyPrice_Stock1'] - data['BuyPrice_Stock2']
data['z_score_pair'] = (ratio - moving_average_43) / moving_average_43.std()
data

In [ ]:
# Initialize positions
positions = pd.Series(index=data.index, data=0)
# Generate buy signals (when z-score is less than -0.05)
positions[data['z_score_pair'] < -0.2] = 1  # Buy position
# Generate sell signals (when z-score is greater than 0.05)
positions[data['z_score_pair'] > 0.2] = -1  # Sell position
# Close all positions (when z-score is equal to 0)
positions[data['z_score_pair'] == 0] = 0  # Close all positions
# Add the positions to the DataFrame
data['position'] = positions
# Display the resulting DataFrame
data

In [ ]:
data.columns

In [ ]:
import pandas as pd
import numpy as np
# Load your dataset (replace this with your actual data loading method)
# Ensure your DataFrame has the columns: 'datetime', 'BuyPrice_Stock1', 'SellPrice_Stock1', 'BuyPrice_Stock2', 'SellPrice_Stock2'
data['datetime'] = pd.to_datetime(data['datetime'])
# Parameters
initial_capital = 100000  # Starting capital
brokerage_fee = 0.0001  # 0.01% brokerage fee
risk_percentage = 1  # 100% risk per trade
ma_window = 5500  # Moving average window
z_score_threshold = 0.2  # Z-score threshold
# Calculate spread and rolling statistics
data['spread'] = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) / (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])
data['rolling_mean'] = data['spread'].rolling(window=ma_window).mean()
data['rolling_std'] = data['spread'].rolling(window=ma_window).std()
data['z_score'] = (data['spread'] - data['rolling_mean']) / data['rolling_std']
# Initialize variables
portfolio_value = initial_capital
current_position = 0
entry_price1, entry_price2 = None, None
entry_datetime = None
cumulative_pnl = 0
tradebook = []
# Simulate trades
for i, row in data.iterrows():
z_score = row['z_score']
sell_price1 = row['SellPrice_Stock1']
buy_price1 = row['BuyPrice_Stock1']
sell_price2 = row['SellPrice_Stock2']
buy_price2 = row['BuyPrice_Stock2']
current_datetime = row['datetime']
# Skip iteration if prices are NaN
if pd.isna(sell_price1) or pd.isna(buy_price2):
continue
# Calculate max risk and position sizes
max_risk = portfolio_value * risk_percentage
position_size1 = round(max_risk / (2 * sell_price1)) if sell_price1 > 0 else 0
position_size2 = round(max_risk / (2 * sell_price2)) if sell_price2 > 0 else 0
# Entry conditions
if z_score > z_score_threshold and current_position == 0:  # Sell stock1, buy stock2
entry_price1, entry_price2 = sell_price1, buy_price2
entry_datetime = current_datetime
current_position = -1  # Short stock1, long stock2
elif z_score < -z_score_threshold and current_position == 0:  # Buy stock1, sell stock2
entry_price1, entry_price2 = buy_price1, sell_price2
entry_datetime = current_datetime
current_position = 1  # Long stock1, short stock2
# Exit condition: Z-score reverts to neutral
elif current_position != 0 and abs(z_score) < 0.01:
if current_position == 1:  # Long position
pnl_long = (position_size1 * (sell_price1 - entry_price1)) + (position_size2 * (entry_price2 - buy_price2))
transaction_cost = brokerage_fee * (position_size1 * (entry_price1 + sell_price1) + position_size2 * (entry_price2 + buy_price2))
pnl_long -= transaction_cost
cumulative_pnl += pnl_long
tradebook.append({
'Entry Price Stock1': entry_price1,
'Entry Price Stock2': entry_price2,
'Exit Price Stock1': sell_price1,
'Exit Price Stock2': buy_price2,
'Trade Type': 'Long',
'Position Size Stock1': position_size1,
'Position Size Stock2': position_size2,
'PnL': round(pnl_long, 2),
'Transaction Cost': round(transaction_cost, 2),
'Portfolio Value': portfolio_value,
'Cumulative PnL': round(cumulative_pnl, 2),
'Entry Date': entry_datetime.date(),
'Entry Time': entry_datetime.time(),
'Exit Date': current_datetime.date(),
'Exit Time': current_datetime.time()
})
elif current_position == -1:  # Short position
pnl_short = (position_size1 * (entry_price1 - sell_price1)) + (position_size2 * (buy_price2 - entry_price2))
transaction_cost = brokerage_fee * (position_size1 * (entry_price1 + sell_price1) +
position_size2 * (entry_price2 + buy_price2))
pnl_short -= transaction_cost
cumulative_pnl += pnl_short
tradebook.append({
'Entry Price Stock1': entry_price1,
'Entry Price Stock2': entry_price2,
'Exit Price Stock1': sell_price1,
'Exit Price Stock2': buy_price2,
'Trade Type': 'Short',
'Position Size Stock1': position_size1,
'Position Size Stock2': position_size2,
'PnL': round(pnl_short, 2),
'Transaction Cost': round(transaction_cost, 2),
'Portfolio Value': portfolio_value,
'Cumulative PnL': round(cumulative_pnl, 2),
'Entry Date': entry_datetime.date(),
'Entry Time': entry_datetime.time(),
'Exit Date': current_datetime.date(),
'Exit Time': current_datetime.time()
})
# Reset position
current_position = 0
# Convert tradebook to DataFrame and export
tradebook_df = pd.DataFrame(tradebook)
tradebook_df.to_csv('tradebook.csv', index=False)
# Display results
print(tradebook_df)

In [ ]:
import pandas as pd
import numpy as np
# Load your dataset (replace this with your actual data loading method)
# Ensure your DataFrame has the columns: 'datetime', 'BuyPrice_Stock1', 'SellPrice_Stock1', 'BuyPrice_Stock2', 'SellPrice_Stock2'
data['datetime'] = pd.to_datetime(data['datetime'])
# Parameters
initial_capital = 100000  # Starting capital
brokerage_fee = 0.0001  # 0.01% brokerage fee
risk_percentage = 1  # 100% risk per trade
ma_windows = [4300, 5500, 5700, 6300, 6500, 6700, 7100, 7300]  # Moving average windows
z_score_thresholds = [ 0.2, 0.3, 0.35, 0.4]  # Z-score thresholds
# Results storage
results = []
# Loop through each combination of MA and Z-score threshold
for ma_window in ma_windows:
# Calculate spread and rolling statistics
data['spread'] = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) / (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])
data['rolling_mean'] = data['spread'].rolling(window=ma_window).mean()
data['rolling_std'] = data['spread'].rolling(window=ma_window).std()
data['z_score'] = (data['spread'] - data['rolling_mean']) / data['rolling_std']
# Iterate through Z-score thresholds
for z_threshold in z_score_thresholds:
# Initialize variables
portfolio_value = initial_capital
current_position = 0
entry_price1, entry_price2 = None, None
entry_datetime = None
cumulative_pnl = 0
tradebook = []
# Simulate trades
for i, row in data.iterrows():
z_score = row['z_score']
sell_price1 = row['SellPrice_Stock1']
buy_price1 = row['BuyPrice_Stock1']
sell_price2 = row['SellPrice_Stock2']
buy_price2 = row['BuyPrice_Stock2']
current_datetime = row['datetime']
# Skip iteration if prices are NaN
if pd.isna(sell_price1) or pd.isna(buy_price2):
continue
# Calculate max risk and position sizes
max_risk = portfolio_value * risk_percentage
position_size1 = round(max_risk / (2 * sell_price1)) if sell_price1 > 0 else 0
position_size2 = round(max_risk / (2 * sell_price2)) if sell_price2 > 0 else 0
# Entry conditions
if z_score > z_threshold and current_position == 0:  # Sell stock1, buy stock2
entry_price1, entry_price2 = sell_price1, buy_price2
entry_datetime = current_datetime
current_position = -1  # Short stock1, long stock2
elif z_score < -z_threshold and current_position == 0:  # Buy stock1, sell stock2
entry_price1, entry_price2 = buy_price1, sell_price2
entry_datetime = current_datetime
current_position = 1  # Long stock1, short stock2
# Exit condition: Z-score reverts to neutral
elif current_position != 0 and abs(z_score) < 0.01:
if current_position == 1:  # Long position
pnl_long = (position_size1 * (sell_price1 - entry_price1)) + (position_size2 * (entry_price2 - buy_price2))
transaction_cost = brokerage_fee * (position_size1 * (entry_price1 + sell_price1) + position_size2 * (entry_price2 + buy_price2))
pnl_long -= transaction_cost
cumulative_pnl += pnl_long
tradebook.append({
'Entry Price Stock1': entry_price1,
'Entry Price Stock2': entry_price2,
'Exit Price Stock1': sell_price1,
'Exit Price Stock2': buy_price2,
'Trade Type': 'Long',
'Position Size Stock1': position_size1,
'Position Size Stock2': position_size2,
'PnL': round(pnl_long, 2),
'Transaction Cost': round(transaction_cost, 2),
'Portfolio Value': portfolio_value,
'Cumulative PnL': round(cumulative_pnl, 2),
'Entry Date': entry_datetime.date(),
'Entry Time': entry_datetime.time(),
'Exit Date': current_datetime.date(),
'Exit Time': current_datetime.time()
})
elif current_position == -1:  # Short position
pnl_short = (position_size1 * (entry_price1 - sell_price1)) + (position_size2 * (buy_price2 - entry_price2))
transaction_cost = brokerage_fee * (position_size1 * (entry_price1 + sell_price1) +
position_size2 * (entry_price2 + buy_price2))
pnl_short -= transaction_cost
cumulative_pnl += pnl_short
tradebook.append({
'Entry Price Stock1': entry_price1,
'Entry Price Stock2': entry_price2,
'Exit Price Stock1': sell_price1,
'Exit Price Stock2': buy_price2,
'Trade Type': 'Short',
'Position Size Stock1': position_size1,
'Position Size Stock2': position_size2,
'PnL': round(pnl_short, 2),
'Transaction Cost': round(transaction_cost, 2),
'Portfolio Value': portfolio_value,
'Cumulative PnL': round(cumulative_pnl, 2),
'Entry Date': entry_datetime.date(),
'Entry Time': entry_datetime.time(),
'Exit Date': current_datetime.date(),
'Exit Time': current_datetime.time()
})
# Reset position
current_position = 0
# Store results
results.append({
'Moving Average Window': ma_window,
'Z-score Threshold': z_threshold,
'Total PnL': cumulative_pnl,
'Total Trades': len(tradebook),
'Final Portfolio Value': portfolio_value,
})
# Convert results to DataFrame and export
results_df = pd.DataFrame(results)
results_df.to_csv('robustness_test_results.csv', index=False)
# Display results
print(results_df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Load the tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')
# Convert 'Exit Date' to datetime format if it's not already
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])
# Add a column for cumulative PnL
tradebook_df['Cumulative PnL'] = tradebook_df['PnL'].cumsum()
# Calculate drawdown
tradebook_df['Max Cumulative PnL'] = tradebook_df['Cumulative PnL'].cummax()
tradebook_df['Drawdown'] = tradebook_df['Cumulative PnL'] - tradebook_df['Max Cumulative PnL']
# Create the plot
fig, ax = plt.subplots(figsize=(12, 6))
# Plot the cumulative PnL
ax.plot(tradebook_df['Exit Date'], tradebook_df['Cumulative PnL'], label='Cumulative PnL', color='blue')
# Fill the drawdown area with red color
ax.fill_between(tradebook_df['Exit Date'], tradebook_df['Cumulative PnL'], tradebook_df['Max Cumulative PnL'], where=(tradebook_df['Cumulative PnL'] < tradebook_df['Max Cumulative PnL']),
facecolor='red', alpha=0.3, label='Drawdown')
# Add labels and title
ax.set_xlabel('Exit Date')
ax.set_ylabel('Cumulative PnL')
ax.set_title('Cumulative PnL and Drawdown Over Time')
ax.legend()
# Rotate x-axis labels for better readability
plt.xticks(rotation=45)
# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Load the tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')
# Convert 'Exit Date' to datetime format if it's not already
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])
# Add a column for cumulative PnL
tradebook_df['Cumulative PnL'] = tradebook_df['PnL'].cumsum()
# Calculate the maximum cumulative PnL up to each point
tradebook_df['Max Cumulative PnL'] = tradebook_df['Cumulative PnL'].cummax()
# Calculate the drawdown as the difference between cumulative PnL and the max cumulative PnL
tradebook_df['Drawdown'] = tradebook_df['Cumulative PnL'] - tradebook_df['Max Cumulative PnL']
# Calculate the average drawdown
average_drawdown = tradebook_df['Drawdown'].mean()
# Create the plot
plt.figure(figsize=(12, 6))
# Plot the drawdown
plt.plot(tradebook_df['Exit Date'], tradebook_df['Drawdown'], label='Drawdown', color='red', linestyle='-', linewidth=2)
# Plot the average drawdown as a horizontal blue line
plt.axhline(y=average_drawdown, color='blue', linestyle='--', label=f'Average Drawdown ({average_drawdown:.2f})')
# Add labels and title
plt.xlabel('Exit Date')
plt.ylabel('Drawdown')
plt.title('Drawdown Over Time')
plt.legend()
# Rotate x-axis labels for better readability
plt.xticks(rotation=45)
# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Load the tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')
# Create a histogram of PnL distribution
plt.figure(figsize=(14, 7))  # Set the size of the plot
plt.hist(tradebook_df['PnL'], bins=50, color='skyblue', edgecolor='black')
# Add labels and title
plt.xlabel('PnL per Trade')
plt.ylabel('Frequency')
plt.title('PnL Distribution of Trades')
# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors as mcolors
# Load your tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')
# Convert 'Entry Date' to datetime format
tradebook_df['Entry Date'] = pd.to_datetime(tradebook_df['Entry Date'])
# Extract Year and Month from 'Entry Date'
tradebook_df['Year'] = tradebook_df['Entry Date'].dt.year
tradebook_df['Month'] = tradebook_df['Entry Date'].dt.month
# Get the first trade of each month for the portfolio value
first_trade_of_month = tradebook_df.groupby(['Year', 'Month']).first().reset_index()
# Merge the first trade portfolio value with the main data
tradebook_df = tradebook_df.merge(first_trade_of_month[['Year', 'Month', 'Portfolio Value']],
on=['Year', 'Month'], suffixes=('', '_FirstTrade'))
# Calculate the Monthly Profit Percentage
tradebook_df['Monthly Profit %'] = (tradebook_df['PnL'] / tradebook_df['Portfolio Value_FirstTrade']) * 100
# Group by Year and Month to get the sum of monthly profit percentages
monthly_profit_df = tradebook_df.groupby(['Year', 'Month'])['Monthly Profit %'].sum().reset_index()
# Pivot the data for the heatmap
heatmap_data = monthly_profit_df.pivot_table(values='Monthly Profit %', index='Year', columns='Month', aggfunc='sum')
heatmap_data.fillna(0, inplace=True)
# Create the custom colormap for positive (green) and negative (red) values
cmap = mcolors.ListedColormap(['#ff6666', '#3BB143'])  # Red for negative, Green for positive
norm = mcolors.TwoSlopeNorm(vmin=heatmap_data.min().min(), vcenter=0, vmax=heatmap_data.max().max())
# Create the heatmap plot
plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap=cmap, norm=norm, cbar_kws={'label': 'Monthly Profit %'}, linewidths=0.5)
# Set plot labels and title
plt.title("Monthly Profit Percentage Heatmap")
plt.xlabel("Month")
plt.ylabel("Year")
plt.xticks(ticks=range(1, 13), labels=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
# Tight layout for better presentation
plt.tight_layout()
# Show the plot on screen
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Load the tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')
# Convert 'Exit Date' to datetime format if it's not already
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])
# Calculate yearly PnL by summing PnL for each year
tradebook_df['Year'] = tradebook_df['Exit Date'].dt.year
yearly_returns = tradebook_df.groupby('Year')['PnL'].sum().reset_index()
# Create a bar chart for yearly returns using Matplotlib
plt.figure(figsize=(14, 6))  # Set the figure size
plt.bar(yearly_returns['Year'], yearly_returns['PnL'], color='skyblue')
# Add title and labels
plt.title('Yearly Returns', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Profit', fontsize=14)
# Show the bar chart
plt.tight_layout()  # Adjust layout to prevent clipping
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Load the tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')
# Convert 'Exit Date' to datetime format if it's not already
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])
# Add a column for cumulative PnL
tradebook_df['Cumulative PnL'] = tradebook_df['PnL'].cumsum()
# Calculate quarterly periods based on the exit date
tradebook_df['Quarter'] = tradebook_df['Exit Date'].dt.to_period('Q')
# Group by quarter and calculate the total profit (sum of PnL) for each quarter
quarterly_profit = tradebook_df.groupby('Quarter')['PnL'].sum()
# Calculate the average quarterly profit
average_profit = quarterly_profit.mean()
# Plot the quarterly profit with grid enabled
plt.figure(figsize=(12, 8))  # Increase figure size
# Plotting the total profit for each quarter
quarterly_profit.plot(kind='bar', color='skyblue', edgecolor='black')
# Add a horizontal line showing the average in red
plt.axhline(y=average_profit, color='red', linestyle='--', label=f'Average: {average_profit:.2f}')
# Add labels and title
plt.xlabel('Quarter', fontsize=14)
plt.ylabel('Quarterly Profit', fontsize=14)
plt.title('Quarterly Profit', fontsize=16)
# Rotate x-axis labels for better readability
plt.xticks(rotation=45, fontsize=12)
# Show grid for better visualization
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
# Add a legend for the average line
plt.legend()
# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors as mcolors
from fpdf import FPDF
# Load the tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')
# Create a FPDF object for PDF generation
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
# Function to save the plot as an image
def save_plot_as_image(plot_function, filename):
plt.figure(figsize=(12, 6))  # Set the size of the plot
plot_function()  # Call the plotting function
plt.tight_layout()  # Adjust layout to prevent clipping
plt.savefig(filename, format='png')  # Save as PNG
plt.close()  # Close the plot to avoid it being displayed in the notebook
# Plot 1: Cumulative PnL and Drawdown Over Time
def plot_cumulative_pnl_and_drawdown():
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])
tradebook_df['Cumulative PnL'] = tradebook_df['PnL'].cumsum()
tradebook_df['Max Cumulative PnL'] = tradebook_df['Cumulative PnL'].cummax()
tradebook_df['Drawdown'] = tradebook_df['Cumulative PnL'] - tradebook_df['Max Cumulative PnL']
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(tradebook_df['Exit Date'], tradebook_df['Cumulative PnL'], label='Cumulative PnL', color='blue')
ax.fill_between(tradebook_df['Exit Date'], tradebook_df['Cumulative PnL'], tradebook_df['Max Cumulative PnL'], where=(tradebook_df['Cumulative PnL'] < tradebook_df['Max Cumulative PnL']),
facecolor='red', alpha=0.3, label='Drawdown')
ax.set_xlabel('Exit Date')
ax.set_ylabel('Cumulative PnL')
ax.set_title('Cumulative PnL and Drawdown Over Time')
ax.legend()
plt.xticks(rotation=45)
# Save the plot and add to PDF
save_plot_as_image(plot_cumulative_pnl_and_drawdown, "cumulative_pnl_and_drawdown.png")
pdf.add_page()
pdf.image("cumulative_pnl_and_drawdown.png", x=10, y=10, w=190)
# Plot 2: Drawdown Over Time
def plot_drawdown():
tradebook_df['Drawdown'] = tradebook_df['Cumulative PnL'] - tradebook_df['Max Cumulative PnL']
average_drawdown = tradebook_df['Drawdown'].mean()
plt.figure(figsize=(12, 6))
plt.plot(tradebook_df['Exit Date'], tradebook_df['Drawdown'], label='Drawdown', color='red', linestyle='-', linewidth=2)
plt.axhline(y=average_drawdown, color='blue', linestyle='--', label=f'Average Drawdown ({average_drawdown:.2f})')
plt.xlabel('Exit Date')
plt.ylabel('Drawdown')
plt.title('Drawdown Over Time')
plt.legend()
plt.xticks(rotation=45)
# Save the plot and add to PDF
save_plot_as_image(plot_drawdown, "drawdown.png")
pdf.add_page()
pdf.image("drawdown.png", x=10, y=10, w=190)
# Plot 3: PnL Distribution of Trades
def plot_pnl_distribution():
plt.figure(figsize=(14, 7))
plt.hist(tradebook_df['PnL'], bins=50, color='skyblue', edgecolor='black')
plt.xlabel('PnL per Trade')
plt.ylabel('Frequency')
plt.title('PnL Distribution of Trades')
# Save the plot and add to PDF
save_plot_as_image(plot_pnl_distribution, "pnl_distribution.png")
pdf.add_page()
pdf.image("pnl_distribution.png", x=10, y=10, w=190)
# Modified function to accept tradebook_df as an argument
def plot_monthly_profit_heatmap(tradebook_df):
tradebook_df['Entry Date'] = pd.to_datetime(tradebook_df['Entry Date'])
tradebook_df['Year'] = tradebook_df['Entry Date'].dt.year
tradebook_df['Month'] = tradebook_df['Entry Date'].dt.month
first_trade_of_month = tradebook_df.groupby(['Year', 'Month']).first().reset_index()
tradebook_df = tradebook_df.merge(first_trade_of_month[['Year', 'Month', 'Portfolio Value']], on=['Year', 'Month'], suffixes=('', '_FirstTrade'))
tradebook_df['Monthly Profit %'] = (tradebook_df['PnL'] / tradebook_df['Portfolio Value_FirstTrade']) * 100
monthly_profit_df = tradebook_df.groupby(['Year', 'Month'])['Monthly Profit %'].sum().reset_index()
heatmap_data = monthly_profit_df.pivot_table(values='Monthly Profit %', index='Year', columns='Month', aggfunc='sum')
heatmap_data.fillna(0, inplace=True)
cmap = mcolors.ListedColormap(['#ff6666', '#3BB143'])
norm = mcolors.TwoSlopeNorm(vmin=heatmap_data.min().min(), vcenter=0, vmax=heatmap_data.max().max())
plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap=cmap, norm=norm, cbar_kws={'label': 'Monthly Profit %'}, linewidths=0.5)
plt.title("Monthly Profit Percentage Heatmap")
plt.xlabel("Month")
plt.ylabel("Year")
plt.xticks(ticks=range(1, 13), labels=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
# Save the plot and add to PDF
save_plot_as_image(lambda: plot_monthly_profit_heatmap(tradebook_df), "monthly_profit_heatmap.png")
pdf.add_page()
pdf.image("monthly_profit_heatmap.png", x=10, y=10, w=190)
# Plot 5: Yearly Returns
def plot_yearly_returns():
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])
tradebook_df['Year'] = tradebook_df['Exit Date'].dt.year
yearly_returns = tradebook_df.groupby('Year')['PnL'].sum().reset_index()
plt.figure(figsize=(14, 6))
plt.bar(yearly_returns['Year'], yearly_returns['PnL'], color='skyblue')
plt.title('Yearly Returns', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Profit', fontsize=14)
# Save the plot and add to PDF
save_plot_as_image(plot_yearly_returns, "yearly_returns.png")
pdf.add_page()
pdf.image("yearly_returns.png", x=10, y=10, w=190)
# Plot 6: Quarterly Profit
def plot_quarterly_profit():
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])
tradebook_df['Quarter'] = tradebook_df['Exit Date'].dt.to_period('Q')
quarterly_profit = tradebook_df.groupby('Quarter')['PnL'].sum()
average_profit = quarterly_profit.mean()
plt.figure(figsize=(12, 8))
quarterly_profit.plot(kind='bar', color='skyblue', edgecolor='black')
plt.axhline(y=average_profit, color='red', linestyle='--', label=f'Average: {average_profit:.2f}')
plt.xlabel('Quarter', fontsize=14)
plt.ylabel('Quarterly Profit', fontsize=14)
plt.title('Quarterly Profit', fontsize=16)
plt.xticks(rotation=45, fontsize=12)
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.legend()
# Save the plot and add to PDF
save_plot_as_image(plot_quarterly_profit, "quarterly_profit.png")
pdf.add_page()
pdf.image("quarterly_profit.png", x=10, y=10, w=190)
# Save the PDF to a file
pdf.output("tradebook_charts.pdf")
print("PDF has been created successfully!")